# Nutosis

It is October 2018. The squirrels in Central Park are running into a problem and we need your help.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help).

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes the train or test set and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preparation should be noted here but actually implemented in the Preparation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit the train/test parquet files and a pickled preprocessor and final model along with these notebooks.

In [1]:
# Imports
import pandas as pd
import numpy as np
import requests
from pathlib import Path
import zipfile

In [2]:
CENSUS_PATH = "data_cache/2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261108.csv"
DISEASED_PATH = "data_cache/diseased.csv"

census = pd.read_csv(CENSUS_PATH)

# diseased.csv has NO header row: the first line is already a squirrel ID,
# so header=None is required (otherwise one diseased squirrel silently disappears)
diseased_ids = set(pd.read_csv(DISEASED_PATH, header=None)[0])

census["date"] = pd.to_datetime(census["Date"].astype(str).str.zfill(8), format="%m%d%Y")
census["diseased"] = census["Unique Squirrel ID"].isin(diseased_ids)

print(f"{len(census):,} sightings | {census['date'].nunique()} distinct days "
      f"({census['date'].min():%b %d} to {census['date'].max():%b %d, %Y})")
print(f"{len(diseased_ids)} IDs in diseased.csv | "
      f"{census['diseased'].sum()} sightings flagged diseased "
      f"({census['diseased'].mean():.1%} of all sightings)")

FileNotFoundError: [Errno 2] No such file or directory: 'data_cache/2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261108.csv'

# Frame the Problem and Look at the Big Picture

## 1. Define the objective in business terms

**Background.** Nutosis is a rabies-like disease in the squirrels of Central Park. Its main symptom is aggression (some reports mention foaming at the mouth and mangy fur, as if the squirrel had been attacked). It is not clear how it spreads, but there is concern it could pass to other animals or to humans. There are **no human cases yet**, so the goal is to contain it before that changes.

**Situation.** The public reports sightings through an app. Animal control can only respond to as many as it has staff for: about **40-50 captures per day**, against **hundreds of sightings per day**. A capture is only confirmed after a blood test (elevated white blood cell count) that takes **a few hours**, so nobody knows whether a trip was worth it until later.

**Costs (from the client)**

- Sending an agent to a sighting costs **\$200**
- Missing a diseased squirrel costs **$1,000** (the cost of a missed opportunity to contain the disease, plus the risk of it spreading).

**What we predict.** For every sighting, the **probability that the squirrel has nutosis**. Animal control uses that probability to decide which squirrels to go out and capture.

**Business objective.** Use the predicted probabilities to choose where to send agents so that the **total cost (trips + missed diseased squirrels) is as low as possible**, within the daily capacity. In plain terms: catch as many diseased squirrels as possible, as early as possible, without wasting trips.

A trip costs **\$200**, and a missed diseased squirrel costs **\$1,000**. So the break-even point is when the probability of disease is greater than 20%:

$$1000 \cdot P(\text{diseased}) > 200 \;\Longleftrightarrow\; P(\text{diseased}) > 20\%.$$

This is why the probability is the right thing to predict: the cost numbers turn it directly into a decision.

**Secondary objective.** Tell the public what to look for, and tell them agents are being sent. Better reports (more useful fields, fewer irrelevant sightings) make narrowing down easier.

## 2. How will your solution be used?

1. A person submits a sighting in the app (checkboxes and free text: what it looks like, fur color, what it was doing, where it was, e.g. rock or tree).
2. The model **scores the sighting as soon as it arrives**. Speed matters because tests take hours and the disease may be spreading.
3. The output is the **probability that the squirrel is diseased**, plus a **ranked list** of the day's sightings sorted by that probability.
4. Dispatchers send agents to the highest-probability sightings, down to the 20% break-even, until the day's capacity (~40-50 captures) is used.
5. The captured squirrel is tested. The result (diseased or not) is **added to the dataset** and the model is retrained as results come in.

**Who uses it:** animal control staff and dispatchers (primary). The client also wants to tell the public what to look for, so the model's findings (which signs matter most) feed into public messaging.

**Mode:** the app is always available and captures never stop, so the system runs continuously. Scoring is real time, and training is repeated as new labeled results arrive.

## 3. What are the current solutions/workarounds (if any)?

* **Today's process:** staff read incoming sightings and only respond to ones that **sound aggressive or odd**. This is manual and depends on how each reporter writes the description, which the client says to take with a grain of salt. There is no probability, only a judgment call.
* **Naive strategies** to beat (computed below from the real data):
  * *Never send an agent*: every diseased squirrel is missed.
  * *Send an agent to every sighting*: every diseased squirrel is caught, but most trips are wasted (and it is impossible with the staff available).
  * *Random sightings up to capacity*: no better than the base rate.
* **Proxy for today's rule:** the census has no "aggression" field, so we approximate it with the closest recorded behaviors (`Approaches` or `Chasing`).

In [3]:
# Costs from the client notes
COST_MISS = 1000   # a diseased squirrel is not captured (spread risk)
COST_TRIP = 200    # every agent trip, whether the squirrel turns out diseased or not

# Sending an agent costs 200 for sure; not sending costs 1000 * P(diseased).
# Sending is worth it when 200 < 1000 * p, i.e. p > 200 / 1000.
BREAK_EVEN_P = COST_TRIP / COST_MISS
print(f"Break-even probability of disease for sending an agent: {BREAK_EVEN_P:.0%}")

def total_cost(trips, missed):
    return COST_TRIP * trips + COST_MISS * missed

def summarize(name, sent):
    # sent: boolean Series, True where an agent is dispatched
    tp = int((sent & census["diseased"]).sum())
    fp = int((sent & ~census["diseased"]).sum())
    fn = int((~sent & census["diseased"]).sum())
    return {"strategy": name, "trips": tp + fp, "caught": tp, "wasted trips": fp,
            "missed": fn, "recall": tp / (tp + fn),
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "total cost ($)": total_cost(tp + fp, fn)}

n_all = pd.Series(True, index=census.index)
never = pd.Series(False, index=census.index)

# Rough stand-in for today's workaround: only respond to "aggressive-sounding" sightings.
# The census has no aggression field, so we use the closest behaviors it records.
aggressive_like = census["Approaches"] | census["Chasing"]

rows = [
    summarize("Never send an agent", never),
    summarize("Send an agent to every sighting", n_all),
    summarize("Current workaround (proxy: approaches or chasing)", aggressive_like),
    summarize("Perfect model (only diseased squirrels)", census["diseased"]),
]
baselines = pd.DataFrame(rows).set_index("strategy")
baselines.round({"recall": 3, "precision": 3})

Break-even probability of disease for sending an agent: 20%


,trips,caught,wasted trips,missed,recall,precision,total cost ($)
strategy,,,,,,,
Never send an agent,0,0,0,379,0.000,NaN,379000
Send an agent to every sighting,3023,379,2644,0,1.000,0.125,604600
Current workaround (proxy: approaches or chasing),449,89,360,290,0.235,0.198,379800
Perfect model (only diseased squirrels),379,379,0,0,1.000,1.000,75800


**What this tells us.** On these two weeks of data:

| Strategy | Total cost |
|---|---|
| Never send an agent (all 379 diseased squirrels missed) | about **\$379,000** |
| Send an agent to every sighting | about **\$604,600** |
| Today's process (proxy) | about **\$379,800** |
| Perfect model (only the diseased squirrels) | about **\$75,800** |

The "aggressive-sounding" proxy costs about the same as never sending anyone, because only about one in five of its trips found a diseased squirrel, which is right at the 20% break-even. The gap between today's cost and a perfect model is large.

*Caveat:* the proxy is a rough guess at today's process, not a measurement.

## 4. How should you frame this problem?

* **Target value: the probability that a squirrel has the disease,** $P(\text{diseased} \mid \text{report})$. This is the number the model outputs for every sighting and the number animal control acts on.
* **Supervised learning.** We have past sightings with a known outcome (diseased or not, from `data_cache/diseased.csv`). The 1/0 outcome is the training label, and the model learns how likely a 1 is.
* **Probabilistic binary classification, one row per sighting.** Features: only what is in the report (age, fur color, location, behaviors, activities, notes, hectare, shift, date), because that is all we know when we have to decide. We do **not** hand out hard "diseased / not diseased" labels. The decision comes afterwards, from the probability and the costs.
* **The probability must be trustworthy (calibrated).** If the model says 30%, about 30% of such squirrels should really be diseased. The 20% break-even only works if that holds, so calibration is a requirement, not a nice-to-have.
* **Ranking and threshold on top of the probability.** Sort the day's sightings by probability, send agents to those above 20%, and stop when capacity runs out.
* **Imbalanced classes.** About 12.5% of sightings are diseased. Plain accuracy is misleading (never sending anyone is "87.5% accurate").
* **Small dataset.** 3,023 sightings and 379 diseased, over about two weeks. This favors simple, regularized models and careful cross-validation.
* **Offline training with frequent retraining, real-time scoring.** New results arrive all the time, so the model should be retrained regularly (for example daily) on all results so far. True online learning is not needed because labels arrive hours later and in modest numbers.
* **Not stationary.** The disease may be spreading, and weather changes how many people report sightings. Validation should respect time (train on earlier days, test on the latest days) rather than shuffle everything.
* **Biased labels (watch out).** In production, only squirrels that staff investigate get a lab result, and staff investigate the ones that sound aggressive. The model could end up learning today's rule. This is something to plan for in later steps.
* *Alternative framings to keep in mind:* a daily **ranking** problem (best 45 sightings of the day), **text classification** of the free-text notes, and **positive-unlabeled learning** if squirrels not in `data_cache/diseased.csv` turn out never to have been tested (see assumption 1).

## 5. How should performance be measured? Is the performance measure aligned with the business objective?

**Business metric: total cost** over a period, when the predicted probabilities are used to decide who gets an agent.

$$\text{cost} = 200 \times (\text{trips}) + 1000 \times (\text{diseased squirrels missed})$$

It is directly comparable to the baselines above and easy to explain. We also track **cost saved vs. the best naive strategy**.

**Machine learning metrics** (all reported on a held-out time period). Because the target is a probability, some metrics judge **how good the probabilities are** and others judge **how well the decisions made from them work**.

| Metric | What it measures | Why |
|---|---|---|
| **Log loss** | Quality of the predicted probabilities | Penalizes confident wrong probabilities heavily, which fits the cost of misses |
| **Brier score** | Quality of the predicted probabilities | Easy to read: average squared gap between probability and outcome |
| **Calibration plot** | Do predicted probabilities match real rates? | We threshold at 20% and may quote probabilities, so they must be trustworthy |
| **Precision-recall AUC** | How well sightings are ranked | Suits imbalanced data; shows whether diseased squirrels sit near the top of the list |
| **Recall** at the chosen threshold | Share of diseased squirrels caught | Missing a diseased squirrel is "VERY bad" |
| **Precision** at the chosen threshold | Share of trips that find a diseased squirrel | Every wasted trip costs $200 and trips are limited. It must stay clearly above 20% |
| **Precision / recall at daily capacity** (top ~45 sightings per day) | Results for the list as it will really be used | Matches the 40-50 captures per day limit |

Accuracy and ROC AUC are not used as headline metrics because of the class imbalance.

**Is the measure aligned with the business objective?** Mostly yes, with these gaps:

* **Good probabilities are not automatically low cost.** A model can have a good log loss and still lose money if the threshold is wrong, so the final check is always the total cost.
* **Capacity is not in the cost formula.** Staff can only do 40-50 captures per day, so a "cost-optimal" model could still ask for more trips than possible. We also report results at capacity.
* **Not all misses are equal.** A flat **\$1,000** ignores that a diseased squirrel missed today can infect others, so missing it early is probably worse than missing it late.
* **"Can't waste trips at all" versus $200.** The notes treat wasted trips as nearly unacceptable but also give them a price. We assume the price is real and the cost is soft (to confirm with the client). If it is a hard limit, the threshold should be higher.
* **Informing the public is not measured** by cost. It is handled qualitatively (which signs the model finds most useful).

## 6. What would be the minimum performance needed to reach the business objective?

**Minimum bar.** On later days not used for training, the model's probabilities must lead to a total cost **lower than both the best naive strategy and today's process**, without needing more than the daily capacity. The probabilities must also be **calibrated**: squirrels given about 20% really are diseased about 20% of the time, because the whole decision rule rests on that number.

**How to read the table below.** At **20% precision**, the cost equals "never send" (about $379,000) **no matter how high recall is**, because each trip then exactly pays for itself. So the squirrels we send agents to must be diseased **clearly more than 20% of the time** for the model to be worth anything. Above that, the higher the recall, the lower the cost.

In [4]:
# What recall and precision does a model need to beat each baseline?
# trips = caught / precision, caught = recall * total_diseased, missed = (1 - recall) * total_diseased
total_diseased = int(census["diseased"].sum())
best_naive = baselines.loc[["Never send an agent", "Send an agent to every sighting"], "total cost ($)"].min()
workaround_cost = baselines.loc["Current workaround (proxy: approaches or chasing)", "total cost ($)"]

recalls = [0.50, 0.70, 0.80, 0.90, 0.95]
precisions = [0.20, 0.30, 0.40, 0.50, 0.70]
grid = pd.DataFrame(
    {f"precision {p:.0%}": [total_cost(r * total_diseased / p, (1 - r) * total_diseased)
                            for r in recalls] for p in precisions},
    index=[f"recall {r:.0%}" for r in recalls])
print(f"Best naive strategy: ${best_naive:,.0f} | current-workaround proxy: ${workaround_cost:,.0f}")

print("Total cost ($) for a given recall / precision:")
display(grid.round(0).map("${:,.0f}".format))   # use .applymap on pandas < 2.1

print("\nDoes it beat BOTH the best naive strategy and the current-workaround proxy?")
display(grid.round(0) < min(best_naive, workaround_cost))

Best naive strategy: $379,000 | current-workaround proxy: $379,800
Total cost ($) for a given recall / precision:


,precision 20%,precision 30%,precision 40%,precision 50%,precision 70%
recall 50%,"$379,000","$315,833","$284,250","$265,300","$243,643"
recall 70%,"$379,000","$290,567","$246,350","$219,820","$189,500"
recall 80%,"$379,000","$277,933","$227,400","$197,080","$162,429"
recall 90%,"$379,000","$265,300","$208,450","$174,340","$135,357"
recall 95%,"$379,000","$258,983","$198,975","$162,970","$121,821"



Does it beat BOTH the best naive strategy and the current-workaround proxy?


,precision 20%,precision 30%,precision 40%,precision 50%,precision 70%
recall 50%,False,True,True,True,True
recall 70%,False,True,True,True,True
recall 80%,False,True,True,True,True
recall 90%,False,True,True,True,True
recall 95%,False,True,True,True,True


* **Speed:** scoring must be immediate (responses are needed within hours).
* **Capacity:** the list must be usable with 40-50 captures per day. This matters on busy days: October 13 had 74 diseased squirrels, more than any team could capture, so even a perfect ranking could not catch all of them that day.

In [5]:
# Daily capacity check: agents can capture ~40-50 squirrels per day (client notes)
CAPACITY_LOW, CAPACITY_HIGH = 40, 50

daily = census.groupby("date").agg(sightings=("diseased", "size"), diseased=("diseased", "sum"))
daily["over capacity (50/day)"] = daily["diseased"] > CAPACITY_HIGH
print(daily)
print(f"\nAverage per day: {daily['sightings'].mean():.0f} sightings, {daily['diseased'].mean():.0f} diseased")
print(f"Days where diseased squirrels exceed {CAPACITY_HIGH} captures: "
      f"{int(daily['over capacity (50/day)'].sum())} of {len(daily)}")

            sightings  diseased  over capacity (50/day)
date                                                   
2018-10-06        337        52                    True
2018-10-07        405        27                   False
2018-10-08        285        38                   False
2018-10-10        335        22                   False
2018-10-12        218        45                   False
2018-10-13        434        74                    True
2018-10-14        368        47                   False
2018-10-17        216        30                   False
2018-10-18        200        14                   False
2018-10-19        158        18                   False
2018-10-20         67        12                   False

Average per day: 275 sightings, 34 diseased
Days where diseased squirrels exceed 50 captures: 2 of 11


## 7. What are comparable problems? Can you reuse experience or tools?

* **Wildlife and animal disease surveillance** (rabies in raccoons and bats, avian influenza, chronic wasting disease in deer): prioritizing which reports to investigate with limited field staff.
* **Medical screening and triage:** risk scores for how likely a patient is to have a condition, where high recall is required and false alarms cost resources, not lives.
* **Alert triage with limited investigators** (fraud detection, security alerts): score alerts with a probability, investigate the top ones, learn from what the investigations find.
* **Citizen-science data quality** (iNaturalist, eBird report validation): noisy volunteer reports.
* **Cost-sensitive classification and threshold tuning:** I can reuse my approach from the Early Spring project, where a similar costs-versus-gain calculation gave a 20% break-even probability. Same tools: scikit-learn pipelines, class weights, `TunedThresholdClassifierCV`, probability calibration (`CalibratedClassifierCV`, calibration curves), and precision-recall curves.
* **Text and spatial features:** TF-IDF for the free-text fields, and location (hectare, latitude/longitude) to spot hot spots.

## 8. Is human expertise available?

* **Bush (client):** owns the labels, knows how the process runs, and sets priorities (costs, capacity, acceptable recall).
* **Animal control staff and dispatchers:** know what an "aggressive-sounding" report looks like and how they decide which ones to respond to.
* **Lab team running the blood tests:** how reliable the diagnosis is and how long it takes.
* **Veterinarians and wildlife disease specialists** (for example NYC Department of Health, NYC Parks, or the Central Park Conservancy): symptoms, how rabies-like diseases spread, and whether fur color or mangy fur is plausibly linked to infection.
* **Squirrel Census team:** how the sighting data and fur colors were recorded.

## 9. How would you solve the problem manually?

A dispatcher reading the day's sightings would probably:

1. Read each report and look for **aggression or odd behavior** (chasing, approaching people, unusual calls, foaming, staggering).
2. Look for **visible signs** such as mangy or patchy fur.
3. Give more weight to **reports from areas with confirmed cases** and to reports that describe a clear behavior over vague ones.
4. Put the most worrying reports first and send agents until the day's capacity runs out.
5. Capture, test, and wait a few hours for the white blood cell count to come back.

In effect, the dispatcher is judging "how likely is this squirrel to be sick?" in their head. The model does the same thing in a measurable way: it learns from the 379 confirmed cases which report details go with disease, weighs them consistently (instead of depending on who reads the report), and outputs an actual probability for each sighting.

A quick look at the data shows the fields do carry signal (not a model, just the share of diseased squirrels in each group):

In [6]:
# A first look at how the label relates to a few report fields (NOT modeling, just framing).
# Reports are user-entered, so we only want to know whether the fields carry any signal at all.
def rate_by(col):
    g = census.groupby(col, dropna=False)["diseased"].agg(["size", "sum", "mean"])
    g.columns = ["sightings", "diseased", "share diseased"]
    return g.round(3)

print(rate_by("Primary Fur Color"), "\n")
print(rate_by("Age"), "\n")
print(rate_by("Approaches"), "\n")
print(rate_by("Shift"))

                   sightings  diseased  share diseased
Primary Fur Color                                     
Black                    103        67           0.650
Cinnamon                 392       188           0.480
Gray                    2473       115           0.047
NaN                       55         9           0.164 

          sightings  diseased  share diseased
Age                                          
?                 4         0           0.000
Adult          2568       329           0.128
Juvenile        330        32           0.097
NaN             121        18           0.149 

            sightings  diseased  share diseased
Approaches                                     
False            2845       329           0.116
True              178        50           0.281 

       sightings  diseased  share diseased
Shift                                     
AM          1347       167           0.124
PM          1676       212           0.126


**What we see.** The share of diseased squirrels differs a lot by fur color (gray about 5%, cinnamon about 48%, black about 65%), and `Approaches` is higher than average (28% vs. 12%). Age and shift look uninformative. This is only a first look and nothing has been tuned on it. The fur-color pattern is strong enough that it should be discussed with an expert before it is trusted (see the questions below). The "mangy fur", "foaming" and "aggression" symptoms from the notes do not appear anywhere in this data (checked below).

## 10. List the assumptions you (or others) have made so far. Verify assumptions if possible.

1. **Labels:** squirrels in `data_cache/diseased.csv` are diagnosed positive, and **every other sighting is negative** (diagnosed not diseased). If non-listed squirrels were never tested, they are unlabeled rather than negative, and the problem becomes positive-unlabeled learning.
2. **The census file stands in for app sightings:** same kind of information (fur color, age, location, behaviors), but the real app may have different fields.
3. **Reports are noisy:** reporters are members of the public and can be wrong ("take with a grain of salt").
4. **Costs:** **\$200** per trip (diseased or not) and $1,000 per missed diseased squirrel, fixed and independent of when or where. This gives the 20% break-even probability.
5. **Capacity:** 40-50 captures per day, with some flexibility.
6. **The diagnosis is the truth:** an elevated white blood cell count means the squirrel is fighting nutosis, with no false positives or negatives.
7. **Two weeks of data is representative of the near future,** even though the disease may spread and weather changes the number of reports.
8. **Only information in the report is usable:** no lab results or other facts known only after capture.
9. **Each sighting is a separate squirrel:** not guaranteed, since several people can report the same animal.
10. **No human health cost is included:** there are no human cases, so only trips and misses are priced.
11. **The decision is per sighting, day by day,** and trips can be sent at any time of day (captures happen at any time).
12. **A well-calibrated probability is achievable** from the report fields alone, with only 379 positive examples.

### Verification

In [7]:
# Verify what we can verify from the data (assumptions above)
checks = {}

# 1. every diseased ID exists in the census
checks["diseased IDs found in census"] = f"{census['diseased'].sum()} of {len(diseased_ids)}"

# 2. the squirrel ID is unique per sighting
dup = census["Unique Squirrel ID"].duplicated(keep=False)
checks["rows sharing an ID with another row"] = f"{int(dup.sum())} (diseased among them: {int(census.loc[dup, 'diseased'].sum())})"

# 3. days with no data between first and last date
all_days = pd.date_range(census["date"].min(), census["date"].max())
missing_days = [d.strftime("%b %d") for d in all_days if d not in set(census["date"])]
checks["calendar days with no sightings"] = ", ".join(missing_days)

# 4. do the free-text fields mention the symptoms in the notes?
text = (census[["Color notes", "Other Activities", "Other Interactions", "Specific Location"]]
        .fillna("").agg(" ".join, axis=1).str.lower())
for kw in ["foam", "mad", "mangy", "aggress", "attack"]:
    checks[f"free-text mentions of '{kw}'"] = int(text.str.contains(kw).sum())

# 5. is there an explicit aggression field?
behavior_cols = ["Running", "Chasing", "Climbing", "Eating", "Foraging", "Kuks", "Quaas",
                 "Moans", "Tail flags", "Tail twitches", "Approaches", "Indifferent", "Runs from"]
checks["behavior columns in the census"] = ", ".join(behavior_cols)
checks["column named aggression/mange/foam"] = any(
    k in c.lower() for c in census.columns for k in ["aggress", "mange", "foam"])

pd.Series(checks, name="result").to_frame()

,result
diseased IDs found in census,379 of 379
rows sharing an ID with another row,10 (diseased among them: 0)
calendar days with no sightings,"Oct 09, Oct 11, Oct 15, Oct 16"
free-text mentions of 'foam',0
free-text mentions of 'mad',52
free-text mentions of 'mangy',0
free-text mentions of 'aggress',0
free-text mentions of 'attack',0
behavior columns in the census,"Running, Chasing, Climbing, Eating, Foraging, ..."
column named aggression/mange/foam,False


| # | Assumption | What we checked | Status |
|---|---|---|---|
| 1 | Listed = diseased, others = negative | All 379 IDs in `data_cache/diseased.csv` match a census sighting, and 379 fits the "few hundred diagnosed" in the notes. But the file cannot show whether the other 2,644 were tested ||
| 2 | Census = app sightings | The census has behaviors and fur color but **no aggression, mangy-fur or foaming field**, and none of those words appears in the free text but the word 'mad' does. These are the main symptoms in the notes |  |
| 3 | Reports are noisy | Cannot be checked with this data. Treated as a reason to prefer simple, robust features | Assumed |
| 4 | Costs and 20% break-even | Taken from the client's notes | Client-provided. Confirm "no wasted trips" is not a hard limit |
| 5 | Capacity of 40-50/day | On 2 of 11 days there were more diseased squirrels (52 and 74) than 50 | Checked: capacity is binding on busy days |
| 6 | Diagnosis is truth | Cannot be checked | Assumed |
| 7 | Two weeks is representative | The diseased share ranges from 7% to 21% across days, so it changes from day to day | Risky: the split is random, so check with cross-validation grouped by day in the Training step.|
| 8 | Report fields only | We only use census columns that exist at report time, and not the ID (it contains the date and hectare, but no label) | Plan for later steps |
| 9 | One sighting = one squirrel | 5 IDs appear twice (10 rows). None are diseased, so the label join is unaffected, but they should be reviewed | Checked, minor |
| 10 | Nothing else priced | No human cases per the notes | Client-provided |
| 11 | Data coverage | No sightings on Oct 9, 11, 15 and 16 in the file | Ask whether those days are missing or just had no data |
| 12 | Calibrated probabilities are achievable | Cannot be checked until modeling. Only 379 positives, so calibration will be noisy | To test in the training step |

# Get the Data

**Goal:** collect everything needed to predict the **probability that a reported squirrel has nutosis**, clean it just enough to use it, and **lock away a test set** before any modelling.

## Why this step matters for the project

Animal control gets a report from the app and has to decide **whether to send an agent**. A trip costs **\$200**, a missed diseased squirrel costs **\$1,000**, so a trip is worth it once the probability of disease is above **20%**. To estimate that probability well, the model needs more than the report itself. This step gathers the report data, the answers (which squirrels were diseased), and two outside sources that add context.

## How the pieces fit together

| Piece | Comes from | Joined to the sightings by | Why we want it |
|---|---|---|---|
| Sightings (what, where, when) | Squirrel census 2018 | n/a, this is the main table | The reports the model will see |
| Label `diseased` | `diseased.csv` from Bush | `Unique Squirrel ID` | What the model learns to predict |
| **Geography** (named places) | USGS Board on Geographic Names | **Each sighting's latitude and longitude** | Squirrels near water, bridges or woods may have different exposure. Names also tell animal control **exactly where to go** |
| **Visitor activity** (weather + calendar) | NOAA daily weather, calendar computed in code | Sighting **date** | Weather and weekends change how many people feed or touch squirrels |

## 1. Data needed and how much

| Data | What it gives us | Amount | Role |
|---|---|---|---|
| Squirrel census (2018) | One row per sighting: location (`X` longitude, `Y` latitude), date, shift, fur, age, behaviour, approaches | 3,023 rows × 31 columns | Features |
| `diseased.csv` (from Bush) | IDs of the 379 squirrels that have nutosis | 379 IDs, no header row | **Label** (target) |
| **External 1:** USGS named places | Named lakes, bridges, hills, woods and buildings in and around the park, with coordinates | One file for New York State, cut down to the park area | Geography features and readable place names |
| **External 2:** NOAA daily weather + calendar | Temperature, rain and wind per day, plus weekend and holiday information | One row per day (about 3 weeks needed) | Visitor-activity features |

**How much is enough?** All 3,023 sightings are used. The external sources are tiny after filtering to the park area and the sighting dates. More rows cannot be added for squirrels, because the census is the only squirrel data that exists.

## 2. Where the data comes from

| Source | Link | Format |
|---|---|---|
| Squirrel census | NYC Open Data, "2018 Central Park Squirrel Census – Squirrel Data" (already in the project folder) | CSV |
| `diseased.csv` | Client file from Bush (already in the project folder) | CSV, one ID per line |
| USGS Domestic Names | https://prd-tnm.s3.amazonaws.com/StagedProducts/GeographicNames/DomesticNames/DomesticNames_NY_Text.zip (also listed at https://www.usgs.gov/us-board-on-geographic-names/download-gnis-data) | Zip with a pipe-separated text file |
| NOAA GHCN-Daily, station USW00094728 (NYC Central Park) | https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/USW00094728.csv | CSV |
| Calendar features | Computed with pandas' US federal holiday calendar | Nothing to download |

## 3. Access and authorizations

| Source | Authorization |
|---|---|
| Census | Public NYC Open Data, free to use |
| `diseased.csv` | Client data from Bush. Used only for this project and **never committed** |
| USGS names | US government data, public domain, no key needed |
| NOAA weather | US government data, public domain, no key needed |

## 4. Workspace

- All downloads go into a `data_cache/` folder next to the notebook, which is created in the following line.

## 5. Get the Data

- A directory called `data_cache/` is created next to the notebook. It is used to **cache the external sources** so they do not need to be downloaded again. The client files are already in the project folder and are **never committed**.

In [8]:
CACHE_DIR = Path("data_cache")
CACHE_DIR.mkdir(exist_ok=True)

This allows us to **re-run the notebook without re-downloading** the external sources, which is important for reproducibility and for working offline. The `fetch_cached` function below handles the download and caching.

In [9]:
HEADERS = {"User-Agent": "nutosis-class-project/1.0 (educational use)"}


def fetch_cached(url: str, filename: str, hint: str = "", refresh: bool = False) -> Path:
    # Download `url` once into data_cache/filename. If it fails (or a zip turns out to be a web page),
    # the error says how to download the file by hand instead.
    path = CACHE_DIR / filename
    if refresh or not path.exists():
        tmp = path.with_name(path.name + ".part")
        try:
            with requests.get(url, headers=HEADERS, stream=True, timeout=120) as resp:
                resp.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
            if filename.endswith(".zip") and not zipfile.is_zipfile(tmp):
                raise ValueError("the answer is not a zip file (probably a web page that blocks scripts)")
        except (requests.RequestException, ValueError) as err:
            tmp.unlink(missing_ok=True)
            raise RuntimeError(f"Could not download {filename}: {type(err).__name__}: {err}\n"
                               f"Manual fallback: open {url} in a browser, save the file as {path} "
                               f"and run the cell again. {hint}") from err
        tmp.replace(path)
    return path

Below, we read the census and diseased files, compute the park area, and download the two external sources. The USGS file is a zip with a pipe-separated text file inside, and the NOAA file is a CSV. Both are cached in `data_cache/` so they do not need to be downloaded again.

In [10]:
# --- Client files (already in the project folder) ---
sightings_raw = pd.read_csv(CENSUS_PATH)
# diseased.csv has NO header row, so header=None is required
diseased_ids = set(pd.read_csv(DISEASED_PATH, header=None)[0])

# Park area = extent of the squirrel sightings plus about 300 m on each side
PAD = 0.003
south, north = sightings_raw["Y"].min() - PAD, sightings_raw["Y"].max() + PAD
west, east = sightings_raw["X"].min() - PAD, sightings_raw["X"].max() + PAD
BOX = (south, north, west, east)

LANDMARKS_CACHE = CACHE_DIR / "landmarks.parquet"     # cleaned copies, these are what we keep
WEATHER_CACHE = CACHE_DIR / "weather.parquet"
names_path = weather_path = None                       # stay None when the cleaned copy already exists

# --- External source #1: named places with coordinates (USGS Board on Geographic Names, New York State) ---
if LANDMARKS_CACHE.exists():
    print("landmarks.parquet already exists, no download needed")
else:
    loose_txt = CACHE_DIR / "DomesticNames_NY.txt"     # fallback: the .txt extracted by hand from the zip
    names_path = loose_txt if loose_txt.exists() else fetch_cached(
        "https://prd-tnm.s3.amazonaws.com/StagedProducts/GeographicNames/DomesticNames/DomesticNames_NY_Text.zip",
        "DomesticNames_NY_Text.zip",
        hint="The same file is listed at https://www.usgs.gov/us-board-on-geographic-names/download-gnis-data "
             "(choose New York, Text). If the zip will not open on your computer, extract the .txt file with the "
             "longest name from it and save it as data_cache/DomesticNames_NY.txt instead.")

# --- External source #2: weather for visitor activity (NOAA daily data, Central Park station) ---
NOAA_STATION = "USW00094728"            # New York City Central Park
if WEATHER_CACHE.exists():
    print("weather.parquet already exists, no download needed")
else:
    weather_path = fetch_cached(
        f"https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/{NOAA_STATION}.csv",
        f"{NOAA_STATION}.csv")

for p in (names_path, weather_path):
    if p is not None:
        print(f"Raw file ready: {p.name} ({p.stat().st_size / 1e6:.1f} MB)")

landmarks.parquet already exists, no download needed
weather.parquet already exists, no download needed


## 6. Convert the data to an easy format (without changing the data)

| Source | Conversion | Values changed? |
|---|---|---|
| Census | Dates like `10062018` turned into real dates in a new `date` column | No, the original `Date` column stays |
| `diseased.csv` | Turned into a boolean `diseased` column by matching `Unique Squirrel ID` | No, label added next to the data |
| USGS names | Pipe-separated text read into a table, coordinates turned into numbers, rows cut down to the park area | No, rows are only filtered. **Names are kept exactly as USGS publishes them** |
| NOAA weather | Only the needed columns (date, rain, max/min temperature, wind) and only the days around the sightings kept | No, rows and columns are only filtered |

### About the place names

- **Names are never shortened or rewritten.** `name` is the official USGS name.
- A `full_name` label is built so a person can find the place without guessing: **name + official type + county + state**, for example `Bow Bridge (Bridge), New York County, NY`. (New York County is Manhattan.)
- **Kinds of place come from the official feature class** (for example Lake → water, Bridge → bridge, Summit → hill, Woods → green). The mapping is written out in the code so anyone can review it.
- The park itself (the entry named "Central Park") is not counted as a landmark.
- **Limit:** USGS decides which places have names. If the file calls something "The Pool", that is the most exact official name available.
- The cleaned names table is saved as `landmarks.parquet` and the cleaned weather as `weather.parquet`.

In [11]:
def parse_census_date(s: pd.Series) -> pd.Series:
    # Census-style dates are integers like 10062018 (MMDDYYYY); fall back to normal parsing.
    s = s.astype(str).str.replace(r"\.0$", "", regex=True)
    if s.str.fullmatch(r"\d{7,8}").all():
        return pd.to_datetime(s.str.zfill(8), format="%m%d%Y")
    return pd.to_datetime(s)


# --- Sightings: same values, real dates and a boolean label added next to them ---
sightings = sightings_raw.copy()
sightings["date"] = parse_census_date(sightings["Date"])
sightings["diseased"] = sightings["Unique Squirrel ID"].isin(diseased_ids)
assert sightings["diseased"].sum() == len(diseased_ids), "some diseased IDs are not in the census"

# --- Named places ---
# Official feature class -> the kind of place it is
CLASS_TO_KIND = {
    "Lake": "water", "Reservoir": "water", "Basin": "water", "Bay": "water", "Stream": "water", "Spring": "water",
    "Swamp": "water", "Canal": "water", "Falls": "water", "Rapids": "water", "Channel": "water", "Island": "water",
    "Bridge": "bridge", "Arch": "bridge", "Tunnel": "bridge", "Crossing": "bridge",
    "Summit": "hill", "Ridge": "hill", "Cliff": "hill", "Slope": "hill", "Pillar": "hill", "Gap": "hill",
    "Bench": "hill", "Valley": "hill",
    "Park": "green", "Woods": "green", "Forest": "green", "Flat": "green", "Plain": "green", "Area": "green",
    "Reserve": "green",
    "Building": "structure", "Tower": "structure", "Locale": "structure",
}


def read_names_file(path: Path) -> pd.DataFrame:
    # Read the pipe-separated text file, either from inside the zip or as a loose .txt file
    if path.suffix.lower() == ".txt":
        return pd.read_csv(path, sep="|", dtype=str, encoding="utf-8-sig", encoding_errors="replace", low_memory=False)
    with zipfile.ZipFile(path) as z:
        assert z.testzip() is None, "the zip is damaged, delete it from data_cache/ and run the cell again"
        # the data file is the biggest .txt in the zip (the others only describe the codes)
        member = max((n for n in z.namelist() if n.lower().endswith(".txt") and "decode" not in n.lower()),
                     key=lambda n: z.getinfo(n).file_size)
        print(f"Reading {member} from the zip")
        with z.open(member) as f:
            return pd.read_csv(f, sep="|", dtype=str, encoding="utf-8-sig", encoding_errors="replace", low_memory=False)


if names_path is None:
    landmarks = pd.read_parquet(LANDMARKS_CACHE)
    print(f"Loaded {len(landmarks)} named places from the cache")
else:
    names_all = read_names_file(names_path)
    names_all.columns = names_all.columns.str.strip().str.lower()
    needed = {"feature_id", "feature_name", "feature_class", "prim_lat_dec", "prim_long_dec"}
    assert needed <= set(names_all.columns), f"unexpected columns: {list(names_all.columns)}"
    print(f"{len(names_all):,} named places in New York State")

    names_all["lat"] = pd.to_numeric(names_all["prim_lat_dec"], errors="coerce")
    names_all["lon"] = pd.to_numeric(names_all["prim_long_dec"], errors="coerce")
    in_box = names_all["lat"].between(south, north) & names_all["lon"].between(west, east)
    landmarks = names_all[in_box].copy()
    landmarks["name"] = landmarks["feature_name"].str.strip()          # the official name, exactly as published
    landmarks["kind"] = landmarks["feature_class"].map(CLASS_TO_KIND)
    landmarks["county"] = landmarks["county_name"].fillna("") if "county_name" in landmarks else ""
    print(f"{len(landmarks)} named places inside the park area, "
          f"{landmarks['kind'].notna().sum()} of a kind we use")

    landmarks = (landmarks[landmarks["kind"].notna() & (landmarks["name"] != "Central Park")]   # the park itself is not a landmark
                 .drop_duplicates("feature_id").reset_index(drop=True))
    assert len(landmarks) > 0, "no usable named places found inside the park area, check the file and the area"

    # Full, readable label that animal control can use as-is, e.g. "Bow Bridge (Bridge), New York County, NY"
    landmarks["full_name"] = (landmarks["name"] + " (" + landmarks["feature_class"] + ")"
                              + np.where(landmarks["county"] != "", ", " + landmarks["county"] + " County", "")
                              + ", NY")
    landmarks = landmarks[["feature_id", "name", "full_name", "feature_class", "kind", "lat", "lon"]]
    landmarks.to_parquet(LANDMARKS_CACHE)

print(landmarks.groupby("kind")["full_name"].agg(count="size", examples=lambda s: "; ".join(s.head(3))).to_string())

# --- Weather: only the columns we need, only the days around the sightings ---
if weather_path is None:
    weather_raw = pd.read_parquet(WEATHER_CACHE)
else:
    weather_raw = pd.read_csv(weather_path, usecols=lambda c: c in {"DATE", "PRCP", "TMAX", "TMIN", "AWND"})
    weather_raw["DATE"] = pd.to_datetime(weather_raw["DATE"])
    weather_raw = weather_raw[weather_raw["DATE"].between(sightings["date"].min() - pd.Timedelta(days=7),
                                                          sightings["date"].max())].reset_index(drop=True)
    weather_raw.to_parquet(WEATHER_CACHE)
print(f"\nWeather days: {len(weather_raw)} | columns: {list(weather_raw.columns)}")

Loaded 12 named places from the cache
       count                                                                                                             examples
kind                                                                                                                             
green      2                                Strawberry Fields (Area), New York County, NY; Great Lawn (Flat), New York County, NY
hill       1                                                                             Great Hill (Summit), New York County, NY
water      9  Harlem Meer (Lake), New York County, NY; The Lake (Lake), New York County, NY; The Pond (Lake), New York County, NY

Weather days: 22 | columns: ['DATE', 'PRCP', 'TMAX', 'TMIN', 'AWND']


Above, the sightings in `landmarks.parquet` and `weather.parquet` are fully read and checked, and the cleaned copies are saved. The raw downloads are only needed to create the cleaned copies, so they can be deleted once the cleaned copies are saved. The next cell does that.

### Clean up the raw downloads

The raw zip (a whole state of place names) and the raw weather file (decades of daily weather) are only needed to create the two small parquet files. Once those are saved and checked, the raw files have **no further use** and are deleted.

- A raw file is only deleted if its cleaned copy exists **and is not empty**. Otherwise it is kept.
- Leftover `.part` files from interrupted downloads are removed too.
- To get a raw file back, delete the matching parquet file in `data_cache/` and run the notebook again, or set `KEEP_RAW_DOWNLOADS = True` before the first run.

In [12]:
# Once the cleaned copies are saved, the raw downloads have no further use, so they can be deleted.
# (Set KEEP_RAW_DOWNLOADS = True to keep them.)
KEEP_RAW_DOWNLOADS = False
RAW_FILES = {
    LANDMARKS_CACHE: [CACHE_DIR / "DomesticNames_NY_Text.zip", CACHE_DIR / "DomesticNames_NY.txt"],
    WEATHER_CACHE: [CACHE_DIR / "USW00094728.csv"],
}
for cleaned, raws in RAW_FILES.items():
    cleaned_ok = cleaned.exists() and len(pd.read_parquet(cleaned)) > 0
    for raw in raws:
        if not raw.exists():
            continue
        if cleaned_ok and not KEEP_RAW_DOWNLOADS:
            size_mb = raw.stat().st_size / 1e6
            raw.unlink()
            print(f"Deleted {raw.name} ({size_mb:.1f} MB), the cleaned copy {cleaned.name} is saved")
        elif not cleaned_ok:
            print(f"Kept {raw.name}: the cleaned copy {cleaned.name} is missing or empty")
for stray in CACHE_DIR.glob("*.part"):
    stray.unlink()                                 # leftovers of an interrupted download
print("Files in data_cache/:", sorted(p.name for p in CACHE_DIR.iterdir()))

Files in data_cache/: ['2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261108.csv', 'diseased.csv', 'landmarks.parquet', 'weather.parquet']


## 7. Sensitive information

| Data | Sensitive? | What we do |
|---|---|---|
| Census coordinates and dates | No, squirrels, not people | Keep |
| **Volunteer initials** in free-text notes (for example "... - JO") | **Yes, they identify the volunteers** | **Removed** from `Color notes`, `Other Activities`, `Other Interactions` and `Specific Location`. The cell counts matches before and after, and also checks for email addresses and phone numbers |
| `diseased.csv` (client data) | Private to the client | Git-ignored, never committed |
| USGS names and NOAA weather | No, public data about places and weather | Keep |

**Why only the end of a note?** Volunteers signed notes with their initials at the end. The pattern only removes 2–3 capital letters after a dash at the end of a note, so normal uppercase text such as "AM" or "NYC" inside a note is left alone. The before and after counts show it worked (4 found, 0 left).

In [13]:
# Free-text fields: remove volunteer initials such as " - JO" at the end of a note
FREE_TEXT = ["Color notes", "Other Activities", "Other Interactions", "Specific Location"]
INITIALS = r"\s*-\s*[A-Z]{2,3}(?=\)?\s*$)"
PATTERNS = {"initials at end of note": INITIALS,
            "email address": r"[\w.+-]+@[\w-]+\.[\w.]+",
            "phone number": r"\(?\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}"}


def count_hits(df: pd.DataFrame) -> pd.Series:
    # number of free-text cells (across the four columns) that match each pattern
    return pd.Series({k: int(sum(df[c].fillna("").str.contains(p, regex=True).sum() for c in FREE_TEXT))
                      for k, p in PATTERNS.items()})


before = count_hits(sightings)
for col in FREE_TEXT:
    sightings[col] = sightings[col].str.replace(INITIALS, "", regex=True)
after = count_hits(sightings)
print(pd.DataFrame({"before": before, "after": after}))

                         before  after
initials at end of note       4      0
email address                 0      0
phone number                  0      0


## 8. Join the external data and check size and type

### How each external source is joined

| Source | Join key | Columns added |
|---|---|---|
| USGS named places | **Each sighting's latitude (`Y`) and longitude (`X`)**. Both are projected to metres, then the distance to every named place is computed | See the data dictionary below |
| NOAA weather | Sighting `date` | `wx_*` columns |
| Calendar | Sighting `date` | `cal_*` columns |

### Data dictionary: columns added in this step

| Column | Meaning | Used by |
|---|---|---|
| `geo_dist_water_m` / `_bridge_m` / `_hill_m` / `_green_m` / `_structure_m` | Metres from the sighting to the nearest named place of that kind | **Model feature** |
| `geo_nearest_place` | Full official label of the nearest named place **if within 150 m**, otherwise `"none"` | **Model feature** |
| `geo_nearest_place_kind` | Kind of that place (water, bridge, ...) or `"none"` | **Model feature** |
| `geo_nearest_place_dist_m` | Metres to the nearest named place, **always filled** | People (directions) |
| `geo_nearest_place_direction` | Compass direction of the sighting as seen from that place (north, north-east, ...) | People (directions) |
| `geo_where_text` | One sentence for animal control, for example `About 40 m north-west of Bow Bridge (Bridge), New York County, NY (lat 40.77, lon -73.97)` | People (directions) |
| `wx_tmax_c`, `wx_tmin_c` | Daily high and low in °C | **Model feature** |
| `wx_prcp_mm`, `wx_prcp_prev_day_mm` | Rain that day and the day before, in mm | **Model feature** |
| `wx_wind_ms` | Average wind in m/s | **Model feature** |
| `cal_day_of_week`, `cal_is_weekend`, `cal_days_to_holiday` | Day of week (Monday = 0), weekend flag, days to the nearest US federal holiday | **Model feature** |

The `geo_where_text`, distance and direction columns are for **animal control to read**, not for the model. They are free text built from the sighting's own position, so they do not use the label.

### Notes on the join

- Yesterday's rain is used (not tomorrow's), because it is known by the time of the sighting.
- NOAA's raw files use tenths of °C and tenths of mm. The code converts them if needed, then checks that temperatures are realistic.
- The merges use `many_to_one` and check that the row count does not change.
- The **direction** is the angle from the named place to the squirrel, rounded to 8 compass points.

### Size and type

A later cell lists every table's size and type: **tabular**, **geographical** (coordinates), and **time series** (daily weather, plus date and AM/PM shift on the sightings).

In [14]:
n = len(sightings)
LAT0, LON0 = (south + north) / 2, (west + east) / 2


def to_xy(lat, lon) -> np.ndarray:
    # Simple local projection to metres (accurate enough inside one park)
    x = (np.asarray(lon) - LON0) * 111_320 * np.cos(np.radians(LAT0))
    y = (np.asarray(lat) - LAT0) * 110_540
    return np.column_stack([x, y])


pts = to_xy(sightings["Y"], sightings["X"])             # each sighting: its own longitude (X) and latitude (Y)
lm_xy = to_xy(landmarks["lat"], landmarks["lon"])


def distances_to(mask) -> np.ndarray:
    # distance in metres from every sighting to every landmark selected by `mask`
    return np.sqrt(((pts[:, None, :] - lm_xy[mask.values][None, :, :]) ** 2).sum(axis=2))


# ---------- Geography: distance to the nearest place of each kind, and the nearest named place ----------
for kind in ["water", "bridge", "hill", "green", "structure"]:
    col = f"geo_dist_{kind}_m"
    mask = landmarks["kind"] == kind
    if mask.sum() == 0:
        print(f"WARNING: no '{kind}' places found for this area, {col} left missing")
        sightings[col] = np.nan
    else:
        sightings[col] = distances_to(mask).min(axis=1).round(1)

# Nearest named place of any kind, with distance and direction, so animal control can find the spot
d = distances_to(pd.Series(True, index=landmarks.index))
nearest = d.argmin(axis=1)
nearest_dist = d.min(axis=1)
full_names = landmarks["full_name"].values
close = nearest_dist <= 150

# Direction of the sighting as seen FROM the place (0 degrees = north, 90 = east)
dx = pts[:, 0] - lm_xy[nearest, 0]
dy = pts[:, 1] - lm_xy[nearest, 1]
bearing = np.degrees(np.arctan2(dx, dy)) % 360
COMPASS = np.array(["north", "north-east", "east", "south-east", "south", "south-west", "west", "north-west"])
direction = COMPASS[np.round(bearing / 45).astype(int) % 8]

# Used as model features: only within 150 m, otherwise "none"
sightings["geo_nearest_place"] = np.where(close, full_names[nearest], "none")
sightings["geo_nearest_place_kind"] = np.where(close, landmarks["kind"].values[nearest], "none")
# Used for people (not as model features): always filled, full official name
sightings["geo_nearest_place_dist_m"] = nearest_dist.round(0)
sightings["geo_nearest_place_direction"] = direction
sightings["geo_where_text"] = [
    (f"At {nm}" if dist < 10 else f"About {dist:.0f} m {dr} of {nm}") + f" (lat {la:.5f}, lon {lo:.5f})"
    for nm, dist, dr, la, lo in zip(full_names[nearest], nearest_dist, direction, sightings["Y"], sightings["X"])]
print(f"Named places used: {len(landmarks)} | sightings within 150 m of one: {close.mean():.0%}")
print("Example of what animal control would read:", sightings["geo_where_text"].iloc[0])

# ---------- Visitor activity: weather + calendar ----------
w = weather_raw.rename(columns={"DATE": "date"}).copy()
if w["TMAX"].max() > 60:                  # raw NOAA units: tenths of a degree C and tenths of mm
    for c in ["TMAX", "TMIN", "PRCP", "AWND"]:
        if c in w:
            w[c] = w[c] / 10
assert -30 < w["TMAX"].max() < 45, "temperatures look wrong, check the units"
w = w.sort_values("date")
w["wx_prcp_prev_day_mm"] = w["PRCP"].shift(1)         # yesterday's rain is known by today
w = w.rename(columns={"TMAX": "wx_tmax_c", "TMIN": "wx_tmin_c", "PRCP": "wx_prcp_mm", "AWND": "wx_wind_ms"})
sightings = sightings.merge(w, on="date", how="left", validate="many_to_one")
assert len(sightings) == n, "join changed the number of sightings"

# Calendar features (computed here, nothing downloaded)
from pandas.tseries.holiday import USFederalHolidayCalendar
holidays = USFederalHolidayCalendar().holidays(start=sightings["date"].min() - pd.Timedelta(days=60),
                                               end=sightings["date"].max() + pd.Timedelta(days=60))
sightings["cal_day_of_week"] = sightings["date"].dt.dayofweek            # Monday = 0
sightings["cal_is_weekend"] = sightings["cal_day_of_week"] >= 5
sightings["cal_days_to_holiday"] = sightings["date"].map(lambda d: int(np.abs((holidays - d).days).min()))

# Drop external columns that have no values at all (no named places of that kind in the park area)
empty_cols = [c for c in sightings.columns if c.startswith("geo_dist_") and sightings[c].isna().all()]
sightings = sightings.drop(columns=empty_cols)
print("Dropped columns with no values at all:", empty_cols)

ext_cols = [c for c in sightings.columns if c.startswith(("geo_", "wx_", "cal_"))]
print("\nExternal columns added:", ext_cols)
print("Share of sightings with a value (not missing):")
print(sightings[ext_cols].notna().mean().round(3).to_string())

Named places used: 12 | sightings within 150 m of one: 19%
Example of what animal control would read: About 375 m south-east of Great Hill (Summit), New York County, NY (lat 40.79408, lon -73.95613)
Dropped columns with no values at all: ['geo_dist_bridge_m', 'geo_dist_structure_m']

External columns added: ['geo_dist_water_m', 'geo_dist_hill_m', 'geo_dist_green_m', 'geo_nearest_place', 'geo_nearest_place_kind', 'geo_nearest_place_dist_m', 'geo_nearest_place_direction', 'geo_where_text', 'wx_prcp_mm', 'wx_tmax_c', 'wx_tmin_c', 'wx_wind_ms', 'wx_prcp_prev_day_mm', 'cal_day_of_week', 'cal_is_weekend', 'cal_days_to_holiday']
Share of sightings with a value (not missing):
geo_dist_water_m               1.000
geo_dist_hill_m                1.000
geo_dist_green_m               1.000
geo_nearest_place              1.000
geo_nearest_place_kind         1.000
geo_nearest_place_dist_m       1.000
geo_nearest_place_direction    1.000
geo_where_text                 1.000
wx_prcp_mm                 

/var/folders/1d/h583rbfj6cx29980b7ysz3000000gp/T/ipykernel_51247/1961183851.py:74: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  sightings["cal_day_of_week"] = sightings["date"].dt.dayofweek            # Monday = 0


In [15]:
def describe(name, df, kind):
    return {"source": name, "rows": len(df), "columns": df.shape[1],
            "memory (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 2), "type": kind}


overview = pd.DataFrame([
    describe("Squirrel sightings (+ label)", sightings, "tabular, geographical (X/Y), time (date + AM/PM)"),
    describe("Named places in the park area (USGS)", landmarks, "geographical (named points)"),
    describe("Weather (NOAA, daily)", weather_raw, "time series (one row per day)"),
]).set_index("source")
print(overview.to_string(), "\n")

print("Column types in the sightings table:")
print(sightings.dtypes.astype(str).value_counts().to_string(), "\n")

inside = sightings["Y"].between(south, north) & sightings["X"].between(west, east)
print(f"Sightings inside the Central Park box: {inside.mean():.1%}")
print(f"Dates: {sightings['date'].min():%b %d} to {sightings['date'].max():%b %d, %Y} "
      f"({sightings['date'].nunique()} days), shifts: {sorted(sightings['Shift'].unique())}")
print(f"Hectares: {sightings['Hectare'].nunique()}")

                                      rows  columns  memory (MB)                                              type
source                                                                                                            
Squirrel sightings (+ label)          3023       49         1.54  tabular, geographical (X/Y), time (date + AM/PM)
Named places in the park area (USGS)    12        7         0.00                       geographical (named points)
Weather (NOAA, daily)                   22        5         0.00                     time series (one row per day) 

Column types in the sightings table:
str               18
bool              15
float64           11
int64              3
datetime64[us]     1
int32              1 

Sightings inside the Central Park box: 100.0%
Dates: Oct 06 to Oct 20, 2018 (11 days), shifts: ['AM', 'PM']
Hectares: 339


## 9. Sample a test set and put it aside

**How the split works**

- **Random 80/20 split**, with a fixed seed (`42`) so the result is reproducible.
- **Stratified by the label:** diseased and healthy squirrels are split separately, so both sets keep the same share of diseased squirrels (about 12.5%).
- **Grouped by `Unique Squirrel ID`:** a few IDs appear on two rows. Both rows always go to the same set, so the same squirrel is never in training and test.
- **Only the ID and the label decide the split.** No other column is used.

**Why random and not "the last few days"**

| Question | Finding |
|---|---|
| How uneven are the days? | The disease rate varies from about 7% to 21% between days. A test set of only the last 4 days has just 74 diseased squirrels and a different rate (11.5% vs 12.8%) |
| What does a random split give? | About 76 diseased squirrels in the test set, the same rate as training, and every day and both shifts represented |
| What is the cost? | Rows from the same day end up in both sets, so day-level features (weather, calendar) can be partly memorized. I measured this on the whole table: knowing only the day improves log loss by about 0.006 (baseline about 0.38), so the effect is small |
| How will this be checked? | In the modelling step, cross-validation **grouped by day** on the training set. It scores the model on days it has not seen, and shows how much the random split flatters the result |

**Safety checks in the code**

1. No squirrel ID appears in both sets.
2. Both sets contain diseased and healthy squirrels.
3. The diseased share is within 0.5 points in train and test, and the test set is about 20% of the rows.

After saving, the code **deletes the test set and the full table from memory** and reloads only the training set.

In [16]:
SEED = 42
TEST_FRACTION = 0.20
rng = np.random.default_rng(SEED)

# One row per squirrel ID, so the few IDs that appear twice stay together on the same side
ids = sightings.groupby("Unique Squirrel ID")["diseased"].agg(label="max", rows="size")

test_ids = []
for label, grp in ids.groupby("label"):                       # split each class separately = stratified
    order = rng.permutation(grp.index.to_numpy())
    cum_rows = grp.loc[order, "rows"].cumsum().to_numpy()
    n_ids = int(np.searchsorted(cum_rows, TEST_FRACTION * grp["rows"].sum())) + 1
    test_ids.extend(order[:n_ids])

is_test = sightings["Unique Squirrel ID"].isin(test_ids)
train, test = sightings[~is_test], sightings[is_test]

# Safety checks
assert not set(train["Unique Squirrel ID"]) & set(test["Unique Squirrel ID"])
assert test["diseased"].nunique() == 2 and train["diseased"].nunique() == 2
assert abs(train["diseased"].mean() - test["diseased"].mean()) < 0.005
assert abs(len(test) / len(sightings) - TEST_FRACTION) < 0.01

train.to_parquet("train_set.parquet", index=False)
test.to_parquet("test_set.parquet", index=False)
print(f"Train: {len(train):,} rows ({len(train)/len(sightings):.0%}), {int(train['diseased'].sum())} diseased")
print(f"Test:  {len(test):,} rows ({len(test)/len(sightings):.0%}), {int(test['diseased'].sum())} diseased")
print(f"Days present in both sets: {len(set(train['date']) & set(test['date']))} of {sightings['date'].nunique()}")

# From here on, work only with the training set. The test set is not opened again until the final evaluation.
del test, sightings, is_test
train = pd.read_parquet("train_set.parquet")
print("Reloaded training set:", train.shape)

Train: 2,418 rows (80%), 303 diseased
Test:  605 rows (20%), 76 diseased
Days present in both sets: 11 of 11
Reloaded training set: (2418, 49)


### Why an 80/20 split is necessary for this problem

**The core reason: the model will be trusted with money.**
Animal control will use the predicted probability to decide where to send agents. Each unnecessary trip costs **\$200** and each missed diseased squirrel costs **\$1,000**. Before anyone relies on the model, we need an honest estimate of how it behaves on squirrels it has **never seen**. A model that is scored on its own training data will look better than it really is, because it can memorize the sightings it was shown. The only way to get an honest score is to hold some sightings back, so they play no part in building the model.

**Why a separate test set is needed here, specifically**

1. **The threshold is a decision, not just a number.** The break-even point is a 20% probability. If the threshold or the model is tuned on the same rows that are used to report the result, the reported cost will be too low. The test set is the only data that was not used for any of those choices.
2. **The result must be comparable and trusted.** The client needs a cost estimate they can believe. A score from data the model never touched is the only kind that supports that.
3. **No data snooping.** The test set is saved and put aside before any modelling, and it is not opened again until the final evaluation. Looking at it earlier, even informally, would let it influence our choices.

**Why 20% for the test set, and not more or less**

The split is a trade-off. A bigger test set gives a more reliable final score, but leaves less data to train on. The key constraint is the **number of diseased squirrels** in each set, because recall and cost depend mostly on them.


- **10% is too small.** With only about 38 diseased squirrels in the test set, one or two squirrels would move the recall by several points, so the final estimate would be too noisy to trust.
- **30% costs too much training data.** Models learn the disease pattern from the diseased examples, and there are few of them. Giving up about 40 more diseased squirrels from training hurts more than the small gain in test precision.
- **20% is the compromise.** It keeps about 76 diseased squirrels in the test set and about 300 for training. 80/20 is a widely used convention, but here it is also supported by these numbers. Even so, the final recall estimate is only accurate to about ±10 points, and the write-up should report it with that uncertainty.

**What this split does not do**

- It does not make the model better. It only makes the evaluation honest.
- It does not choose the threshold or the model. Those choices are made on the training set using cross-validation, and the test set only confirms the result.
- It cannot remove the limits of the data itself: 11 observation days in one month and a small number of diseased squirrels.

## Assumptions, data snooping note and outputs

**Assumptions**
- The census coordinates (`X` longitude, `Y` latitude) are accurate enough to measure distance to named places within a few tens of metres.
- The date-level weather of the Central Park station is a fair stand-in for the weather at every sighting.
- The USGS feature class is a reasonable label for the kind of place.
- The official USGS name is specific enough for animal control to find the place. This is checked by reading the example sentences.

**Data snooping note**
- In the Frame step I printed summary tables on **all rows** (fur colour, age, approaches, shift, daily counts). Any pattern noticed there (for example fur colour) is only a **hypothesis**. It must be re-checked on the **training set only** before it is used.
- The test set has not been looked at in this step.
- “The split justification used disease rates per day on the whole table, before the test set was set aside. Only the sizes and class balance of the test set were checked after the split.”

**Glossary**

| Term | Meaning |
|---|---|
| Sighting | One row of the census: one squirrel seen at one place and time |
| Label / target | `diseased`, whether the squirrel has nutosis |
| Feature | A column the model may use to predict the label |
| Data snooping | Looking at the test set (or letting it influence choices) before the final evaluation |
| Feature class | USGS's official type of a named place (Lake, Bridge, Summit, ...) |
| Parquet | A compact table file format that keeps column types |

**Outputs of this step**

| File | Committed to git? | Purpose |
|---|---|---|
| `train_set.parquet` | Yes | Training data with the label and all external columns |
| `test_set.parquet` | Yes | Sealed test set, opened only for the final evaluation |
| `data_cache/landmarks.parquet`, `data_cache/weather.parquet` | No, git-ignored | Cleaned copies of the external data, so later runs need no internet |
| Raw zip and raw weather CSV | No, **deleted after cleaning** | Only needed to build the two parquet files above |
| Census CSV and `diseased.csv` | No, git-ignored | Original client and census files |

**Questions to add to the single message to Bush**
1. Does the app record the reporter's own coordinates, or the squirrel's? This changes how far we can trust the location data.
2. Is a place name plus distance and direction enough for an agent to find the spot, or does animal control prefer coordinates, a map pin or park landmarks of their own?